# Markview — render living-painting cinemagraphs (Colab T4)

Generates seamless looping MP4s for each atmosphere painting using Stable Video Diffusion XT. Runs free on a Colab T4 (15 GB VRAM) in ~30 s per painting.

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Free tier is fine.

1. Cells 1–2 install deps + clone the repo (no auth, public).
2. Cell 3 loads SVD-XT (~9 GB download, one-time).
3. Cell 4 renders all 14 paintings.
4. Cell 5 downloads them as a zip. Extract into `apps/web/public/atmospheres/` in your local checkout, commit, deploy.

In [ ]:
# 1) Install deps + ffmpeg (ffmpeg already on Colab; pin diffusers).
!pip install -q --upgrade diffusers transformers accelerate imageio imageio-ffmpeg
!ffmpeg -version | head -1

In [ ]:
# 2) Clone the markview repo so we have the source paintings in place.
import os, shutil
if os.path.exists('markview'):
    shutil.rmtree('markview')
!git clone --depth 1 https://github.com/abgnydn/markview.git
PAINTINGS_DIR = 'markview/apps/web/public/atmospheres'
import pathlib
jpgs = sorted(pathlib.Path(PAINTINGS_DIR).glob('*.jpg'))
print(f'found {len(jpgs)} paintings')
for j in jpgs: print(' ', j.name)

In [ ]:
# 3) Load SVD-XT on the T4 (~9 GB weights download, one-time).
import torch
from diffusers import StableVideoDiffusionPipeline
MODEL = 'stabilityai/stable-video-diffusion-img2vid-xt'
pipe = StableVideoDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, variant='fp16').to('cuda')
pipe.enable_model_cpu_offload()
try: pipe.enable_vae_slicing()
except Exception: pass
try: pipe.enable_vae_tiling()
except Exception: pass
print('pipeline ready')

In [ ]:
# 4) Render each painting -> seamless looping MP4 next to the JPG.
#    SVD-XT does 25 frames @ 6fps = 4.16s clip; ffmpeg xfade closes
#    the loop by cross-fading the last 8 frames into the first 8.
import subprocess, pathlib, torch
from diffusers.utils import load_image, export_to_video

TARGET_W, TARGET_H = 1024, 576
FRAMES, FPS = 25, 6
MOTION_DEFAULT = 110
# Per-painting motion tuning. Mountain-anchored scenes use low motion
# so the subject stays still; water / snow / wheat-field use more.
MOTION_OVERRIDES = {
    'fuji-hokusai': 70, 'fuji-36492': 80, 'fuji-56216': 90, 'fuji-56240': 90,
    'wave-hokusai': 140, 'wave-36501': 130, 'wave-56238': 130, 'wave-53700': 130,
    'snow-hiroshige': 110, 'snow-57043': 130,
    'fields-vangogh': 110, 'fields-335537': 110, 'fields-437980': 100, 'fields-437998': 100,
}

for src in sorted(pathlib.Path(PAINTINGS_DIR).glob('*.jpg')):
    dst = src.with_suffix('.mp4')
    if dst.exists():
        print(f'[skip] {src.name}')
        continue
    motion = MOTION_OVERRIDES.get(src.stem, MOTION_DEFAULT)
    print(f'\n[render] {src.name} (motion={motion})')
    image = load_image(str(src)).convert('RGB').resize((TARGET_W, TARGET_H))
    out = pipe(
        image,
        decode_chunk_size=8,
        generator=torch.manual_seed(42),
        motion_bucket_id=motion,
        noise_aug_strength=0.02,
        num_frames=FRAMES,
        fps=FPS,
    )
    raw = src.with_name(src.stem + '.raw.mp4')
    export_to_video(out.frames[0], str(raw), fps=FPS)
    # Seamless-loop xfade in ffmpeg.
    xfade_secs = 8.0 / FPS
    full_secs  = FRAMES / FPS
    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw),
        '-filter_complex',
        f'[0:v]split=2[a][b];'
        f'[b]trim=0:{xfade_secs},setpts=PTS-STARTPTS+{full_secs - xfade_secs}/TB[b2];'
        f'[a][b2]xfade=transition=fade:duration={xfade_secs}:offset={full_secs - xfade_secs}',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
        '-preset', 'slow', '-crf', '23', str(dst),
    ], check=True)
    raw.unlink()
    print(f'[done] {dst.name} ({dst.stat().st_size // 1024} KB)')
print('\nall paintings rendered.')

In [ ]:
# 5) Zip the MP4s + download. Extract the zip into your local
#    apps/web/public/atmospheres/ folder, commit, deploy.
import shutil, pathlib
from google.colab import files
out = pathlib.Path('cinemagraphs')
if out.exists(): shutil.rmtree(out)
out.mkdir()
for mp4 in pathlib.Path(PAINTINGS_DIR).glob('*.mp4'):
    shutil.copy(mp4, out / mp4.name)
zip_path = shutil.make_archive('markview-cinemagraphs', 'zip', out)
print(f'zipped {len(list(out.iterdir()))} MP4s -> {zip_path}')
files.download(zip_path)